:important: This is the notebook for my Wildfires project for SDS210. Broad structure as follows:

1. Import required packages
    

2. test pull data into project with the FIRMS API and then just use the bounding box for Australia.

3. reproject to correct CRS


4. 

In [1]:
import requests
import pandas as pd
import time

In [2]:
# We need to access the API and to do that, will use the map key that permits access.
MAP_KEY = '54684dde74a099b139ddbbef0f621891'

# Now let's check how many results we have

url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  response = requests.get(url)
  data = response.json()
  df = pd.Series(data)
  display(df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object

In [3]:
# let's create a simple function that tells us how many transactions we have used.
# We will use this in later examples

def get_transaction_count() :
  count = 0
  try:
    response = requests.get(url)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

tcount = get_transaction_count()
print ('Our current transaction count is %i' % tcount)

Our current transaction count is 0


In [4]:
# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
df = pd.read_csv(da_url)
display(df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-02-01,2026-05-11
1,MODIS_SP,2000-11-01,2026-01-31
2,VIIRS_NOAA20_NRT,2026-03-01,2026-05-11
3,VIIRS_NOAA20_SP,2018-04-01,2026-02-28
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-11
5,VIIRS_SNPP_NRT,2026-03-01,2026-05-11
6,VIIRS_SNPP_SP,2012-01-20,2026-02-28
7,LANDSAT_NRT,2022-06-20,2026-05-10
8,GOES_NRT,2022-08-09,2026-05-11
9,BA_MODIS,2000-11-01,2026-02-01


In [5]:
# now let's see how many transactions we use by querying this end point

start_count = get_transaction_count()
pd.read_csv(da_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

# now remember, after 10 minutes this will reset


We used 5 transactions.


In [6]:
# in this example let's look at VIIRS NOAA-20, entire world and the most recent day
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'
start_count = get_transaction_count()
df_area = pd.read_csv(area_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

df_area

We used 36 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,19.40142,-155.28870,346.90,0.42,0.45,2026-05-11,1,N20,VIIRS,l,2.0NRT,317.25,4.43,D
1,19.40241,-155.28065,350.06,0.42,0.45,2026-05-11,1,N20,VIIRS,l,2.0NRT,328.95,13.71,D
2,19.40290,-155.27666,367.00,0.42,0.45,2026-05-11,1,N20,VIIRS,h,2.0NRT,340.95,21.64,D
3,19.40338,-155.27271,367.00,0.42,0.45,2026-05-11,1,N20,VIIRS,h,2.0NRT,341.46,21.64,D
4,19.40387,-155.26875,353.63,0.42,0.45,2026-05-11,1,N20,VIIRS,n,2.0NRT,327.84,17.25,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8268,18.10048,-94.34725,335.72,0.60,0.50,2026-05-11,726,N20,VIIRS,n,2.1URT,272.55,3.99,N
8269,18.10463,-94.34125,305.27,0.60,0.50,2026-05-11,726,N20,VIIRS,n,2.1URT,275.85,1.81,N
8270,18.10525,-94.34663,315.09,0.60,0.50,2026-05-11,726,N20,VIIRS,n,2.1URT,274.37,3.99,N
8271,18.11085,-94.39555,306.54,0.60,0.50,2026-05-11,726,N20,VIIRS,n,2.1URT,290.36,1.71,N


In [7]:
# We can also focus on a smaller area ex. South Asia and get the last 3 days of records
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/54,5.5,102,40/3'
df_area = pd.read_csv(area_url)
df_area

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,18.25584,101.37791,332.06,0.55,0.68,2026-05-09,542,N20,VIIRS,n,2.0NRT,287.33,5.10,D
1,20.51955,101.70763,325.54,0.47,0.64,2026-05-09,544,N20,VIIRS,n,2.0NRT,286.34,5.00,D
2,20.79094,101.56647,330.79,0.48,0.64,2026-05-09,544,N20,VIIRS,n,2.0NRT,285.39,4.70,D
3,20.80259,101.56403,339.16,0.48,0.64,2026-05-09,544,N20,VIIRS,n,2.0NRT,289.47,8.44,D
4,20.80945,101.54031,331.00,0.48,0.64,2026-05-09,544,N20,VIIRS,n,2.0NRT,288.34,4.33,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10788,24.50456,56.60486,321.94,0.71,0.75,2026-05-10,2257,N20,VIIRS,n,2.0NRT,298.03,5.24,N
10789,24.97033,55.63091,311.65,0.61,0.71,2026-05-10,2257,N20,VIIRS,n,2.0NRT,290.05,1.29,N
10790,25.90085,54.53302,320.98,0.50,0.65,2026-05-10,2257,N20,VIIRS,n,2.0NRT,293.54,3.04,N
10791,25.97543,56.07275,308.48,0.61,0.71,2026-05-10,2257,N20,VIIRS,n,2.0NRT,298.45,3.54,N
